In [1]:
# Fabric notebook source
# METADATA ********************
# MPI accuracy evaluation.
#
# Measures the master patient index against the ground truth: which source
# records genuinely belong to the same person. Without this, "the MPI ran"
# is the only claim available. With it, the claim becomes a number.
#
# This is possible only because the data is synthetic. In production nobody
# has the answer key, which is precisely why the review queue and the
# confidence bands exist — they are the operational substitute for a
# measurement you cannot take.
#
# Requires patient_truth.csv in lh_governance/Files/truth/, and the
# governance lakehouse attached alongside lh_bronze.
# ****************************

# PARAMETERS CELL ********************
truth_path = "abfss://8b54538c-e5d1-489c-a821-a0c92c6eb81f@onelake.dfs.fabric.microsoft.com/21884c11-60b6-41c9-9083-01d10eefd761/Files/_truth/patient_truth/ingest_date=2026-08-01/patient_truth_2026-08-01.csv"
# ***********************************

# CELL ********************
from pyspark.sql import functions as F

# MARKDOWN ********************
# ## 1. Load ground truth
#
# patient_truth.csv has one row per real person, with a pipe-delimited list
# of the source ids that person appears under, each prefixed by its system.
# Explode it into (record_uid, true_person_id) pairs so it can be joined to
# the xref the MPI produced.
# ****************************

# CELL ********************
truth_raw = spark.read.option("header", True).csv(truth_path)
print(f"Truth rows (real people): {truth_raw.count():,}")

truth = (truth_raw
    .select("true_person_id", F.explode(F.split("source_ids", r"\|")).alias("sid"))
    .filter(F.col("sid") != "")
    # source_ids arrive as "EHR:MRN12345678"; record_uid in the xref is
    # "EHR|MRN12345678". Same information, different separator.
    .withColumn("record_uid",
                F.concat_ws("|",
                            F.split(F.col("sid"), ":").getItem(0),
                            F.split(F.col("sid"), ":").getItem(1)))
    .select("true_person_id", "record_uid"))

n_truth_records = truth.count()
print(f"Truth record mappings: {n_truth_records:,}")

# MARKDOWN ********************
# ## 2. Join to the MPI output
# ****************************

# CELL ********************
xref = spark.table("silver_patient_xref").select("record_uid", "patient_golden_id")
print(f"MPI record mappings: {xref.count():,}")

joined = truth.join(xref, "record_uid", "inner").cache()
matched = joined.count()
print(f"Records present in both: {matched:,} "
      f"({100 * matched / max(1, n_truth_records):.2f}% of truth)")

unmatched = n_truth_records - matched
if unmatched:
    print(f"WARNING: {unmatched:,} truth records absent from the MPI. "
          f"These are records the pipeline never ingested, not matching errors.")

# MARKDOWN ********************
# ## 3. Pairwise precision and recall
#
# The standard record-linkage measure. Consider every pair of records that
# belong to the same real person, and every pair the MPI grouped together:
#
#   True positive  — same person, and the MPI grouped them
#   False positive — different people, but the MPI merged them
#   False negative — same person, but the MPI kept them apart
#
# The two errors are not equally bad. A false negative fragments a record
# and someone eventually notices a gap. A false positive puts one patient's
# allergies on another patient's chart. Precision matters more than recall
# here, which is why the auto-link threshold sits well above the point that
# would maximise F1.
# ****************************

# CELL ********************
# Pairs the truth says belong together
t = joined.select("true_person_id", F.col("record_uid").alias("uid"))
truth_pairs = (t.alias("x").join(t.alias("y"), "true_person_id")
                .filter(F.col("x.uid") < F.col("y.uid"))
                .select(F.col("x.uid").alias("uid_a"), F.col("y.uid").alias("uid_b")))

# Pairs the MPI grouped together
m = joined.select("patient_golden_id", F.col("record_uid").alias("uid"))
mpi_pairs = (m.alias("x").join(m.alias("y"), "patient_golden_id")
              .filter(F.col("x.uid") < F.col("y.uid"))
              .select(F.col("x.uid").alias("uid_a"), F.col("y.uid").alias("uid_b")))

tp = truth_pairs.intersect(mpi_pairs).count()
fp = mpi_pairs.subtract(truth_pairs).count()
fn = truth_pairs.subtract(mpi_pairs).count()

precision = tp / max(1, tp + fp)
recall = tp / max(1, tp + fn)
f1 = 2 * precision * recall / max(1e-9, precision + recall)

print(f"True positives  : {tp:,}   correctly linked pairs")
print(f"False positives : {fp:,}   wrongly merged  <- clinical safety risk")
print(f"False negatives : {fn:,}   missed links    <- fragmented records")
print()
print(f"Precision : {precision:.4f}   of pairs the MPI merged, this fraction were right")
print(f"Recall    : {recall:.4f}   of pairs that should be merged, this fraction were found")
print(f"F1        : {f1:.4f}")

# MARKDOWN ********************
# ## 4. Cluster-level accuracy
#
# Pairwise metrics can look healthy while individual patients are still
# wrong, so also count how many real people were reconstructed exactly.
# ****************************

# CELL ********************
per_person = (joined.groupBy("true_person_id")
              .agg(F.countDistinct("patient_golden_id").alias("golden_ids"),
                   F.count("*").alias("records")))

per_golden = (joined.groupBy("patient_golden_id")
              .agg(F.countDistinct("true_person_id").alias("true_people"),
                   F.count("*").alias("records")))

people = per_person.count()
split = per_person.filter(F.col("golden_ids") > 1).count()
merged_wrong = per_golden.filter(F.col("true_people") > 1).count()
exact = people - split

print(f"Real people in scope        : {people:,}")
print(f"Reconstructed exactly       : {exact:,} ({100 * exact / max(1, people):.2f}%)")
print(f"Split across multiple ids   : {split:,} ({100 * split / max(1, people):.2f}%)")
print(f"Golden ids holding >1 person: {merged_wrong:,}  <- the dangerous errors")

# MARKDOWN ********************
# ## 5. Where the errors are
#
# An aggregate number is a grade. The breakdown below is what you would act
# on: which attribute was missing when the MPI failed.
# ****************************

# CELL ********************
std_cols = spark.table("silver_patient_golden")

if merged_wrong:
    print("Golden ids containing more than one real person:")
    bad = per_golden.filter(F.col("true_people") > 1).limit(20)
    (bad.join(joined, "patient_golden_id")
        .join(truth_raw.select(F.col("true_person_id"), "given_name", "family_name",
                               "birth_date", "hcn"), "true_person_id")
        .orderBy("patient_golden_id")
        .select("patient_golden_id", "true_person_id", "record_uid",
                "given_name", "family_name", "birth_date")
        .show(40, truncate=False))

if split:
    print("Real people split across multiple golden ids (sample):")
    (per_person.filter(F.col("golden_ids") > 1).limit(20)
        .join(joined, "true_person_id")
        .join(truth_raw.select("true_person_id", "given_name", "family_name",
                               "birth_date", "hcn"), "true_person_id")
        .orderBy("true_person_id")
        .select("true_person_id", "patient_golden_id", "record_uid",
                "given_name", "family_name", "birth_date", "hcn")
        .show(40, truncate=False))

# MARKDOWN ********************
# ## 6. What the review queue would have caught
#
# The pairs sent for human review are the ones the algorithm declined to
# decide. If a large share of the false negatives are sitting in that queue,
# the thresholds are doing their job — the misses are visible rather than
# silent.
# ****************************

# CELL ********************
if spark.catalog.tableExists("silver_patient_match_review"):
    review = spark.table("silver_patient_match_review").select("uid_a", "uid_b")
    n_review = review.count()
    missed = truth_pairs.subtract(mpi_pairs)
    caught = missed.intersect(review).count()
    print(f"Review queue size            : {n_review:,}")
    print(f"False negatives in the queue : {caught:,} of {fn:,} "
          f"({100 * caught / max(1, fn):.1f}%)")
    print()
    print("A high share here means the algorithm knew what it did not know.")
    print("A low share means those links were rejected outright, which is")
    print("the failure mode worth tuning for.")

# MARKDOWN ********************
# ## 7. Result
# ****************************

# CELL ********************
print("=" * 62)
print("MPI ACCURACY")
print("=" * 62)
print(f"  Source records evaluated : {matched:,}")
print(f"  Real people              : {people:,}")
print(f"  Golden patients produced : {joined.select('patient_golden_id').distinct().count():,}")
print()
print(f"  Pairwise precision : {precision:.4f}")
print(f"  Pairwise recall    : {recall:.4f}")
print(f"  Pairwise F1        : {f1:.4f}")
print()
print(f"  People reconstructed exactly : {100 * exact / max(1, people):.2f}%")
print(f"  False merges (golden id spanning people) : {merged_wrong:,}")
print("=" * 62)

import json
mssparkutils.notebook.exit(json.dumps({
    "precision": round(precision, 4),
    "recall": round(recall, 4),
    "f1": round(f1, 4),
    "exact_pct": round(100 * exact / max(1, people), 2),
    "false_merges": merged_wrong,
    "split_people": split,
}))

StatementMeta(, 3f03b9cf-4fc9-4dc6-91c9-d1c2ac6f4be8, 3, Finished, Available, Finished, False)

Truth rows (real people): 25,000
Truth record mappings: 62,431
MPI record mappings: 62,424
Records present in both: 62,431 (100.00% of truth)
True positives  : 47,305   correctly linked pairs
False positives : 12   wrongly merged  <- clinical safety risk
False negatives : 4,595   missed links    <- fragmented records

Precision : 0.9997   of pairs the MPI merged, this fraction were right
Recall    : 0.9115   of pairs that should be merged, this fraction were found
F1        : 0.9536
Real people in scope        : 25,000
Reconstructed exactly       : 22,241 (88.96%)
Split across multiple ids   : 2,759 (11.04%)
Golden ids holding >1 person: 7  <- the dangerous errors
Golden ids containing more than one real person:
+------------------------------------+--------------+-----------------+----------+-----------+----------+
|patient_golden_id                   |true_person_id|record_uid       |given_name|family_name|birth_date|
+------------------------------------+--------------+-------------

In [1]:
for t in ["pharm_medication_order", "facil_hospital", "facil_department", "facil_bed"]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
for t in [x.tableName for x in spark.sql("SHOW TABLES").collect() if x.tableName.startswith("stg_")]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

StatementMeta(, 1373c82a-6feb-4cc7-a4ac-59c58faf8db9, 3, Finished, Available, Finished, False)